# 🧪 Hotel Reservation System 5: Testing the System

A multi-agent system fails in quiet ways: the coordinator misreads a request, skips the compliance check, or
forwards too early. This notebook adds a safeguard and a way to catch regressions:
- the coordinator **rates its own confidence** that it understood the guest, and when that is low it apologises
  and asks the guest to repeat instead of forwarding anything;
- **tests** that check the routing and the verdicts in code: fast unit tests for the router, and a scripted
  table of end-to-end cases.

## Learning Objectives
1. **Extra state** - A `confidence` field next to `messages`, written by the coordinator
2. **Structured self-assessment** - `with_structured_output` returns the score as an integer, never text to parse
3. **Confidence-gated routing** - A pure router that stops forwarding when confidence is low
4. **Router unit tests** - Checking routing decisions on hand-built states, with no LLM calls
5. **A regression table** - Scripted requests with expected verdicts and routes, checked by observing which nodes ran

## Where this fits
`04_hotel_edge_cases_and_errors.ipynb` hardened the core system against specific edge cases. This notebook
adds a confidence score to the core from `03_hotel_compliance_checker.ipynb` (imported from
`hotel_common`) and tests it.
`06_hotel_parallel_reservation_agents.ipynb` then scales the system to parallel agents over two hotels.

## Prerequisites
- The core system from `03_hotel_compliance_checker.ipynb` (read it first; this notebook imports it from `hotel_common.py`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import sqlite3
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel, Field

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by all agents and the self-assessment
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧰 Part 1: Data and Tools

In [ ]:
# ============================================================================
# DATABASE, SQL TOOLS AND THE COMPLIANCE RETRIEVER
# ============================================================================
hc.init_hotel_db("hotel.db")
sql_tools = hc.make_sql_tools("hotel.db")  # 02's tools: run_query only reads, book_room is the only write
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]
print(f"✅ {len(sql_tools)} SQL tools, 1 retrieval tool over {len(hc.COMPLIANCE_RULES)} rules")

---
## 🎯 Part 2: A Coordinator That Rates Its Own Confidence

The state gets two fields next to `messages`: `confidence` (0–100) and the model's one-line reason. On each new
guest message the coordinator first asks the model for a `SelfAssessment` (structured output, so the score is an
`int` and cannot fail to parse). Then:
- **confidence ≥ 50**: it replies as usual (and may forward to an agent);
- **confidence < 50**: it gets an extra instruction to apologise, say what was unclear and ask the guest to repeat.

The coordinator is re-entered after the compliance verdict within the same turn; the score is only computed when
the last message is the guest's, so one guest message gets one assessment.

The threshold is `hc.CONFIDENCE_THRESHOLD` (50), shared with notebook 09. It matches the assessor's scale
below: under 50 the model cannot tell what the guest wants, while 50–79 only means details are missing, which
the coordinator asks for anyway.

In [ ]:
# ============================================================================
# STATE, SELF-ASSESSMENT AND THE COORDINATOR
# ============================================================================
CONFIDENCE_THRESHOLD = hc.CONFIDENCE_THRESHOLD  # 50: hotel_common.py says why
AGENTS = ["reservation_assistant", "compliance_checker"]
conversation_prompt = hc.conversation_prompt(AGENTS, extra_guidelines=hc.COMPLIANCE_FIRST)  # 03's guideline


class HotelState(MessagesState):
    confidence: int          # 0-100, the coordinator's self-assessment of the latest guest message
    confidence_reason: str


class SelfAssessment(BaseModel):
    confidence: int = Field(description="0 to 100: how sure you are that you understand what the guest wants")
    reason: str = Field(description="One short sentence explaining the score")


assessor = llm.with_structured_output(SelfAssessment)
ASSESS_PROMPT = SystemMessage(content="""You monitor a hotel's customer-support desk. Rate from 0 to 100 how confident
you are that you understand what the guest wants in their LATEST message, given the conversation so far.
- 80-100: the intent is clear (a greeting, a question about the hotel, a booking request, a clear off-topic question)
- 50-79: the intent is clear but details are missing (e.g. a booking without dates)
- 0-49: you cannot tell what they want (garbled text, references to things never mentioned, contradictions)""")
UNSURE = SystemMessage(content="You are not sure what the guest wants. Apologise briefly, say what was unclear, and "
                       "ask them to repeat or rephrase their request. Do not name or forward to any agent.")


def conv_assistant(state: HotelState) -> dict:
    update = {}
    if state["messages"][-1].type == "human":  # a new guest message: assess it once
        assessment = assessor.invoke([ASSESS_PROMPT] + state["messages"])
        update = {"confidence": max(0, min(100, assessment.confidence)), "confidence_reason": assessment.reason}
    confidence = update.get("confidence", state.get("confidence", 100))
    extra = [UNSURE] if confidence < CONFIDENCE_THRESHOLD else []
    return {**update, "messages": [llm.invoke([conversation_prompt] + state["messages"] + extra)]}


print("✅ HotelState, SelfAssessment and conv_assistant defined")

### The rest of the core, and a confidence-aware router
The compliance checker, `generate` and the reservation agent come from `hc.make_core_nodes()`: notebook 03's
agents with 04's structured verdict (`ComplianceVerdict`), so tests can compare it exactly. The router is
`hc.route_by_mention()` (03's router and loop guard) with one new rule, checked first: **low confidence never
forwards**, whatever the reply says. The routing decision stays a pure function of the state, which is what makes
it unit-testable in Part 4.

In [ ]:
# ============================================================================
# COMPLIANCE CHECKER, RESERVATION AGENT, ROUTER AND GRAPH
# ============================================================================
sql_assistant, rag_assistant, generate = hc.make_core_nodes(llm, sql_tools, rag_tools)
latest_verdict = hc.latest_verdict  # 'COMPLIANT' / 'VIOLATION' since the latest guest message, or None


def choose_next_node(state: HotelState) -> Literal["reservation_assistant", "compliance_checker", "__end__"]:
    if state.get("confidence", 100) < CONFIDENCE_THRESHOLD:
        return END  # the coordinator asked the guest to repeat: forward nothing
    return hc.route_by_mention(state["messages"], AGENTS) or END  # 03's router, with its loop guard


builder = StateGraph(HotelState)
builder.add_node("conv_assistant", conv_assistant)
builder.add_node("reservation_assistant", sql_assistant)
builder.add_node("compliance_checker", rag_assistant)
builder.add_node("generate", generate)
builder.add_node("sql_tools", ToolNode(sql_tools))
builder.add_node("rag_tools", ToolNode(rag_tools))

builder.add_edge(START, "conv_assistant")
builder.add_conditional_edges("conv_assistant", choose_next_node, ["reservation_assistant", "compliance_checker", END])
builder.add_conditional_edges("reservation_assistant", tools_condition, {"tools": "sql_tools", END: END})
builder.add_conditional_edges("compliance_checker", tools_condition, {"tools": "rag_tools", END: "conv_assistant"})
builder.add_edge("sql_tools", "reservation_assistant")
builder.add_edge("rag_tools", "generate")
builder.add_edge("generate", "conv_assistant")

graph = builder.compile(checkpointer=InMemorySaver())
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## 🔍 Part 3: A Clear Request and an Ambiguous One

`hc.run_traced()` streams a turn with `stream_mode=["updates", "values"]`: each update names a node that ran, so
it reports **which nodes ran**, not just what was said, and the last value is the final state. The ambiguous
request refers to a "last time" that never happened on this fresh thread.

In [ ]:
# ============================================================================
# ONE CLEAR AND ONE AMBIGUOUS REQUEST, TRACED NODE BY NODE
# ============================================================================
stay = date.today() + timedelta(days=60)
CLEAR = f"Hi! Is a Double room free from {stay} to {stay + timedelta(days=3)}?"
AMBIGUOUS = "Same as last time, but move it to the other weekend and make it the bigger one."

for label, text, should_be_confident in (("clear", CLEAR, True), ("ambiguous", AMBIGUOUS, False)):
    nodes, state = hc.run_traced(graph, text, thread_id=f"confidence-{label}", verbose=False)
    confident = state["confidence"] >= CONFIDENCE_THRESHOLD
    print(f"👤 {text}")
    print(f"🎯 confidence {state['confidence']} ({state['confidence_reason']})")
    print(f"🧭 nodes: {' -> '.join(nodes)}")
    print(f"🤵 {state['messages'][-1].content}")
    ok = confident == should_be_confident and (("compliance_checker" in nodes) == should_be_confident)
    print(f"{'✅' if ok else '❌'} {label}: confident={confident}, compliance check ran={'compliance_checker' in nodes}\n")

**Reading the output:** the clear request should score high and go through the usual path (the compliance
checker, then possibly the reservation agent). The ambiguous one should score below 50, run only
`conv_assistant`, and end with an apology and a request to repeat. Compare the two `reason` lines: they say *why*
the model was unsure, which is useful to log.

A self-assessed score is a heuristic, not a calibrated probability. Its value is as a cheap trigger for "ask
again" behaviour; the tests below check that it triggers when it should.

---
## 🧩 Part 4: Unit Tests for the Router (no LLM)

Because `choose_next_node` is a pure function of the state, its rules can be tested on hand-built states in
milliseconds, with no API calls. These tests pin down the loop guard and the confidence rule. The last case is a
**known issue** kept on purpose: string routing is fooled by a reply that merely mentions an agent (notebook 01
showed it). Recording it as an expected failure documents the gap, and the test flags it if a change (for example a
typed router) fixes it.

In [ ]:
# ============================================================================
# ROUTER UNIT TESTS
# ============================================================================
guest = HumanMessage(content="I'd like to book a Deluxe room.")
verdict = AIMessage(content="COMPLIANT: no rule applies.", name="compliance")


def reply(text: str) -> AIMessage:
    return AIMessage(content=text)


ROUTER_TESTS = [
    # (description, state, expected route, known issue?)
    ("confident, forwards to the checker", {"messages": [guest, reply("Forwarding to compliance_checker.")], "confidence": 90},
     "compliance_checker", False),
    ("confident, verdict given, forwards to reservations",
     {"messages": [guest, verdict, reply("Passing you to the reservation_assistant.")], "confidence": 90},
     "reservation_assistant", False),
    ("loop guard: checker named again after a verdict",
     {"messages": [guest, verdict, reply("Let me ask the compliance checker.")], "confidence": 90}, END, False),
    ("old verdict does not count for a new guest message",
     {"messages": [guest, verdict, reply("Done."), guest, reply("compliance_checker")], "confidence": 90},
     "compliance_checker", False),
    ("low confidence never forwards",
     {"messages": [guest, reply("Sorry, I'm unsure. reservation_assistant")], "confidence": 20}, END, False),
    ("plain answer ends the turn", {"messages": [guest, reply("We have a gym on floor 2.")], "confidence": 95},
     END, False),
    ("KNOWN ISSUE: a mere mention routes",
     {"messages": [guest, verdict, reply("I can't involve the reservation assistant until you give me dates.")],
      "confidence": 70}, END, True),
]

passed = 0
for description, state, expected, known_issue in ROUTER_TESTS:
    got = choose_next_node(state)
    if got == expected:
        passed += 1
        print(f"✅ {description:<52} -> {got}")
    else:
        print(f"{'⚠️ ' if known_issue else '❌'} {description:<52} -> {got} (expected {expected})")
print(f"\n{passed}/{len(ROUTER_TESTS)} passed; known issues are expected to fail: this series keeps string routing on purpose (notebook 01, Part 6)")

**Reading the output:** every rule except the known issue should pass. These tests are deterministic: if one
fails after a change to the router, the change broke it, not the model.

---
## 📋 Part 5: A Scripted Regression Table (end to end)

Unit tests cannot tell whether the *model* still behaves. End-to-end cases can: each row is a guest request with
the outcome we expect, and `observe()` extracts the actual outcome from the state and the nodes that ran:
- `confident`: was the self-assessment at or above the threshold?
- `verdict`: the compliance verdict for this request (`None` = not checked)
- `reservation`: did the reservation agent run?
- `booked`: for the booking case, is there exactly one row for the guest in the database?

Every case runs on its own thread, and the database is rebuilt first so the suite gives the same starting point
every run. Keep a table like this next to the system and re-run it after every prompt or model change.

In [ ]:
# ============================================================================
# THE TEST TABLE, RUN AND CHECKED
# ============================================================================
hc.init_hotel_db("hotel.db")  # same starting data for every run of the suite
stay = date.today() + timedelta(days=60)
TEST_CASES = [
    {"case": "general question", "request": "Do you have any rooms with a jacuzzi?",
     "confident": True, "verdict": "COMPLIANT", "reservation": False},
    {"case": "another guest's data", "request": "Which dates is Bob Johnson staying with you?",
     "confident": True, "verdict": "VIOLATION", "reservation": False},
    {"case": "booking, dates missing", "request": "I'd like to book a Suite, please.",
     "confident": True, "verdict": "COMPLIANT", "reservation": False},
    {"case": "complete booking", "request": f"I'm Dana Lee and I'd like to book a Deluxe room from {stay} to "
     f"{stay + timedelta(days=2)}.", "confident": True, "verdict": "COMPLIANT", "reservation": True, "booked": "Dana Lee"},
    {"case": "ambiguous", "request": AMBIGUOUS,
     "confident": False, "verdict": None, "reservation": False},
]


def observe(state: dict, nodes: list[str], case: dict) -> dict:
    actual = {
        "confident": state["confidence"] >= CONFIDENCE_THRESHOLD,
        "verdict": latest_verdict(state["messages"]),
        "reservation": "reservation_assistant" in nodes,
    }
    if "booked" in case:
        with sqlite3.connect("hotel.db") as conn:
            rows = conn.execute("SELECT COUNT(*) FROM reservations WHERE guest_name = ?", (case["booked"],)).fetchone()[0]
        actual["booked"] = case["booked"] if rows == 1 else None
    return actual


report = []
for i, case in enumerate(TEST_CASES, start=1):
    nodes, state = hc.run_traced(graph, case["request"], thread_id=f"test-{i}", verbose=False)
    actual = observe(state, nodes, case)
    failures = [f"{k}: expected {case[k]}, got {v}" for k, v in actual.items() if case[k] != v]
    report.append((case["case"], failures))
    print(f"{'✅' if not failures else '❌'} {case['case']:<24} confidence={state['confidence']:>3} | "
          f"{' -> '.join(nodes)}")
    for failure in failures:
        print(f"      {failure}")

passed = sum(not failures for _, failures in report)
print(f"\n{passed}/{len(report)} cases passed")

**Reading the output:** each line shows the case, the confidence score and the path the request took through the
graph. A ❌ lists exactly which expectation broke. Typical findings:
- `verdict: expected COMPLIANT, got None`: the coordinator skipped the compliance check that the prompt demands;
- `reservation: expected False, got True`: it forwarded a booking before it had dates;
- `booked: expected Dana Lee, got None`: the reservation agent checked availability but asked a question instead
  of booking.

LLM outputs vary, so treat a single ❌ as a lead to investigate and track the pass rate over several runs. What
matters is that the checks are **code**, so the same table can gate a prompt or model change.

To chat with the system yourself, run `hc.chat_with_agent(graph, "my-thread")` in a new cell.

---
## 📝 Summary

- **Extra state** (`confidence`, `confidence_reason`) sits next to `messages` in a `MessagesState` subclass.
- **`with_structured_output(SelfAssessment)`** returns the score as an `int`, so there is no text to parse.
- The **router checks confidence first**: a low score ends the turn with an apology instead of forwarding.
- **Router unit tests** run on hand-built states with no LLM calls, including a documented known issue.
- A **regression table** checks verdicts, routes and bookings end to end by observing which nodes ran
  (`stream_mode="updates"`) and querying the database.

### Next Steps
- `06_hotel_parallel_reservation_agents.ipynb`: parallel reservation agents over two hotels